# VAE — Auto-Encoding Variational Bayes (2013)

_Papers · Generative Models_

**Encode each input to a Gaussian, sample a code through a reparameterization, and maximize a lower bound on the data's likelihood.**

---

This notebook is a practice scaffold for the **VAE — Auto-Encoding Variational Bayes (2013)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the lesson's worked example. ---
mu0, logvar0, eps0 = 0.8, -1.386, -1.2
std0 = torch.tensor(logvar0).mul(0.5).exp().item()          # sigma = exp(0.5*logvar) = 0.5
z0   = mu0 + std0 * eps0                                     # Eqn. 4 reparameterization
# KL (App. B) as the POSITIVE divergence: -0.5*(1 + logvar - mu^2 - sigma^2)
kl0  = -0.5 * (1 + logvar0 - mu0**2 - (std0**2))
print(f"worked example:  sigma={std0:.3f}  z={z0:.3f}  D_KL={kl0:.3f}")
# worked example:  sigma=0.500  z=0.200  D_KL=0.638


# --- 1. The VAE: encoder (mu, logvar) -> reparameterize -> decoder. ---
class VAE(nn.Module):
    def __init__(self, in_dim=784, hidden=400, latent=20):
        super().__init__()
        self.fc1   = nn.Linear(in_dim, hidden)              # encoder body
        self.fc_mu = nn.Linear(hidden, latent)              # head -> mu
        self.fc_lv = nn.Linear(hidden, latent)              # head -> log(sigma^2)
        self.fc3   = nn.Linear(latent, hidden)              # decoder body
        self.fc4   = nn.Linear(hidden, in_dim)              # decoder -> pixels

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_lv(h)                 # mu, logvar

    def reparameterize(self, mu, logvar):                   # Eqn. 4: z = mu + sigma*eps
        std = torch.exp(0.5 * logvar)                       # sigma = exp(0.5*logvar)
        eps = torch.randn_like(std)                         # eps ~ N(0, I)  (the fixed noise)
        return mu + std * eps                               # differentiable in mu, logvar

    def decode(self, z):
        return torch.sigmoid(self.fc4(F.relu(self.fc3(z)))) # pixel probabilities in [0,1]

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


# --- 2. The ELBO loss = reconstruction (summed BCE) + analytic KL (App. B). use_kl toggles the ablation. ---
def elbo_loss(recon, x, mu, logvar, use_kl=True):
    recon_term = F.binary_cross_entropy(recon, x, reduction="sum")   # -E[log p(x|z)], summed over pixels
    kl_term    = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())  # +D_KL (App. B)
    return recon_term + (kl_term if use_kl else 0.0)


# --- 3. MNIST (torchvision is preinstalled in Colab). Flatten to 784, scale to [0,1]. ---
tfm = T.Compose([T.ToTensor()])
full = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
subset = torch.utils.data.Subset(full, range(12000))        # small + fast
loader = torch.utils.data.DataLoader(subset, batch_size=128, shuffle=True)


def train_vae(use_kl=True, epochs=5, latent=20):
    torch.manual_seed(0)
    net = VAE(latent=latent).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    net.train()
    for ep in range(epochs):
        tot = 0.0; n = 0
        for xb, _ in loader:
            xb = xb.view(xb.size(0), -1).to(device)         # (B, 784)
            recon, mu, logvar = net(xb)
            loss = elbo_loss(recon, xb, mu, logvar, use_kl=use_kl)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += xb.size(0)
        print(f"  epoch {ep}  avg -ELBO/img {tot/n:.2f}")
    return net


print("Training VAE (full ELBO: reconstruction + KL):")
vae = train_vae(use_kl=True)

# --- 4. Reconstructions: encode a real batch, decode it back. ---
vae.eval()
xb, _ = next(iter(loader))
xb = xb.view(xb.size(0), -1).to(device)
with torch.no_grad():
    recon, _, _ = vae(xb)
rec_err = F.binary_cross_entropy(recon, xb, reduction="sum").item() / xb.size(0)
print(f"\nReconstruction BCE per image (lower = closer rebuild): {rec_err:.2f}")

# --- 5. Prior sample: draw z ~ N(0, I) and decode -- the real generative test. ---
with torch.no_grad():
    z_prior = torch.randn(16, 20).to(device)                # straight from the prior, no encoder
    samples = vae.decode(z_prior)
# brightness as a crude 'is it a digit?' proxy: trained-with-KL samples have ink, not blank/noise
print(f"Prior-sample mean pixel (with KL): {samples.mean().item():.3f}")

# --- 6. ABLATION: same model, KL term removed. Prior samples should degrade. ---
print("\nTraining ABLATION (reconstruction ONLY, no KL term):")
vae_nokl = train_vae(use_kl=False)
vae_nokl.eval()
with torch.no_grad():
    samples_nokl = vae_nokl.decode(torch.randn(16, 20).to(device))
print(f"Prior-sample mean pixel (NO KL):   {samples_nokl.mean().item():.3f}")
print("With KL the encoded codes cluster near N(0,I), so prior draws decode to digit-like images.")
print("Without KL the codes scatter; prior draws land off-manifold and decode to mush.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)
# To SEE the images in Colab, add:  import matplotlib.pyplot as plt
#   plt.imshow(samples[0].view(28,28).cpu(), cmap="gray"); plt.show()

## Visualize the data & results

_Does the KL term make the latent space samplable? Compare prior samples from a VAE trained WITH vs WITHOUT the KL term._

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision, torchvision.transforms as T

# Reproduce the qualitative effect: the KL term keeps encoded codes near the prior.
torch.manual_seed(0)
full = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=T.ToTensor())
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(full, range(12000)),
                                     batch_size=128, shuffle=True)

class VAE(nn.Module):
    def __init__(self, latent=20):
        super().__init__()
        self.fc1, self.mu, self.lv = nn.Linear(784,400), nn.Linear(400,latent), nn.Linear(400,latent)
        self.fc3, self.fc4 = nn.Linear(latent,400), nn.Linear(400,784)
    def encode(self, x):
        h = F.relu(self.fc1(x)); return self.mu(h), self.lv(h)
    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.exp(0.5*lv) * torch.randn_like(lv)
        return torch.sigmoid(self.fc4(F.relu(self.fc3(z)))), mu, lv

def run(use_kl):
    torch.manual_seed(0); net = VAE(); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for ep in range(5):
        for xb,_ in loader:
            xb = xb.view(xb.size(0),-1)
            recon, mu, lv = net(xb)
            recon_term = F.binary_cross_entropy(recon, xb, reduction="sum")
            kl = -0.5*torch.sum(1 + lv - mu.pow(2) - lv.exp())
            loss = recon_term + (kl if use_kl else 0.0)
            opt.zero_grad(); loss.backward(); opt.step()
    # summarize encoded codes over the data
    net.eval(); mus, sigs = [], []
    with torch.no_grad():
        for xb,_ in loader:
            mu, lv = net.encode(xb.view(xb.size(0),-1))
            mus.append(mu.abs()); sigs.append(torch.exp(0.5*lv))
    mu_abs = torch.cat(mus).mean().item(); sig = torch.cat(sigs).mean().item()
    active = (torch.cat(sigs).mean(0) < 0.8).float().mean().item()  # dims the encoder uses
    return mu_abs, sig, active

for use_kl in (True, False):
    m, s, a = run(use_kl)
    print(("WITH KL " if use_kl else "NO KL   "),
          f"mean|mu|={m:.2f}  mean sigma={s:.2f}  frac active dims={a:.2f}")
# WITH KL : codes stay near N(0,I); prior samples decode to digits.
# NO KL   : codes blow up, sigma -> 0 (deterministic AE); prior samples are off-manifold mush.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Your VAE trains and produces decent prior samples ($z\sim N(0,I)$ &rarr;
            plausible digits). Now delete the KL term from the loss, keep only reconstruction, retrain,
            and again draw $z\sim N(0,I)$ and decode. What happens to the prior samples, and what does that
            prove the KL term is for?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change the loss from recon + kl to just recon; keep architecture, optimizer, data, and seed identical. — _An honest ablation changes exactly one thing &mdash; the KL term &mdash; so any difference is attributable to it._
- Watch reconstructions stay fine (maybe sharper) but prior samples turn to mush or blank. — _Without KL the encoder is free to scatter each input's code anywhere it likes; the codes no longer cluster near $N(0,I)$, so a code drawn from the prior lands in an empty region the decoder never saw._
- Conclude the KL term is what makes the latent space samplable, turning a plain autoencoder into a generative model. — _Reconstruction alone gives an autoencoder; the KL regularizer is the ingredient that aligns the encoded codes with the prior you generate from._

**Answer:** Reconstructions stay good but prior samples collapse &mdash; decoding $z\sim N(0,I)$
                 gives noise or blanks, because without the KL term the encoded codes no longer sit near the
                 prior, so prior draws fall in regions the decoder never trained on. This isolates the KL term
                 as the thing that makes the latent space continuous and samplable: it is what separates a VAE
                 from a plain autoencoder. The CODEVIZ panel shows exactly this contrast.

</details>

**Problem 2.** Your encoder outputs $\mu = 0.8$ and $\log\sigma^2 = -1.386$ for one latent dimension, and you
            draw $\epsilon = -1.2$. Compute the sampled code $z$ and the KL divergence $D_{KL}$ for this
            dimension, and say which way training will nudge $\mu$ and $\sigma$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Get $\sigma$: $\sigma = e^{\frac{1}{2}\log\sigma^2} = e^{-0.693} = 0.5$. — _The encoder predicts logvar; the standard deviation is the exp of half of it._
- Reparameterize (Eqn. 4): $z = \mu + \sigma\epsilon = 0.8 + 0.5(-1.2) = 0.2$. — _The differentiable sample: shift the fixed noise $\epsilon$ by $\mu$ and scale by $\sigma$._
- KL (App. B): $D_{KL} = -\tfrac{1}{2}(1 + \log\sigma^2 - \mu^2 - \sigma^2) = -\tfrac{1}{2}(1 - 1.386 - 0.64 - 0.25) = -\tfrac{1}{2}(-1.276) = 0.638$. — _Plug $\mu^2=0.64$, $\sigma^2=0.25$, $\log\sigma^2=-1.386$ into the closed form; it comes out positive._

**Answer:** $z = 0.2$ and $D_{KL} \approx 0.638$. The divergence is positive because the code
                 distribution ($\mu=0.8$, $\sigma=0.5$) is off-center and too narrow versus the prior
                 $N(0,1)$. The KL gradient pushes $\mu$ toward $0$ and $\sigma$ toward $1$ (where
                 $D_{KL}=0$), while the reconstruction term resists, keeping $z$ informative. These are the
                 worked-example numbers recomputed in the notebook.

</details>

**Problem 3.** A teammate predicts $\sigma$ directly from the encoder and computes the KL as
            $-\tfrac{1}{2}\sum(1 + \sigma^2 - \mu^2 - \sigma^2)$. Two things are wrong. Find them and give
            the correct setup.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot the variance/log-variance mix-up: the formula needs $\log\sigma^2$ in the “$+\log\sigma^2$” slot, but they put $\sigma^2$ there. So the term $1 + \sigma^2 - \mu^2 - \sigma^2$ wrongly cancels to $1 - \mu^2$. — _Appendix B is $\tfrac{1}{2}\sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$: a $\log\sigma^2$ term and a separate $-\sigma^2$ term. Using $\sigma^2$ for both makes them cancel and drops the variance penalty entirely._
- Spot the parameterization issue: predicting $\sigma$ directly needs a positivity constraint (e.g. softplus) and is numerically touchy; predicting $\log\sigma^2$ is unconstrained and stable. — _Standard practice is a logvar head, then $\sigma = e^{\frac{1}{2}\,\text{logvar}}$ for the sample and $\sigma^2 = e^{\text{logvar}}$ for the KL._
- Write it right: kl = -0.5 * sum(1 + logvar - mu.pow(2) - logvar.exp()). — _This is Appendix B exactly: $\log\sigma^2$ is logvar, $\sigma^2$ is logvar.exp(), $\mu^2$ is mu.pow(2)._

**Answer:** Bug 1: they used $\sigma^2$ where the formula needs $\log\sigma^2$, so the
                 “$+\log\sigma^2$” and “$-\sigma^2$” terms cancel and the variance
                 penalty vanishes. Bug 2: predicting $\sigma$ directly is unconstrained-unsafe; predict
                 $\log\sigma^2$ (logvar) instead. Correct KL:
                 -0.5 * sum(1 + logvar - mu.pow(2) - logvar.exp()), with
                 $\sigma = e^{\frac{1}{2}\,\text{logvar}}$ used only for the reparameterized sample.

</details>